# Baseline e Thompson Sampling contextual

O notebook compara a regra fixa conservadora `no_email` com Thompson Sampling Beta-Bernoulli. A avaliação usa replay do experimento randomizado e reserva o teste para a medição final.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.insert(0, str(Path.cwd() / 'src'))
from datathon.data import load_clean, stratified_split
from datathon.evaluation import fixed_policy_value, policy_value, sequential_replay
from datathon.features import FEATURE_COLUMNS
from datathon.policies import ThompsonSamplingPolicy

frame = load_clean(Path('data/raw/hillstrom.csv'))
train, validation, test = stratified_split(frame, seed=2026)
baseline_action = 'no_email'
best_fixed_action = train.groupby('action')['conversion'].mean().idxmax()
baseline_value = fixed_policy_value(test, baseline_action)
best_fixed_value = fixed_policy_value(test, best_fixed_action)
print({'train': len(train), 'validation': len(validation), 'test': len(test), 'baseline_value_ips': baseline_value, 'best_fixed_action': best_fixed_action, 'best_fixed_value_ips': best_fixed_value})


In [ ]:
replay_policy = ThompsonSamplingPolicy(seed=2026)
for row in train.itertuples(index=False):
    values = row._asdict()
    replay_policy.update({column: values[column] for column in FEATURE_COLUMNS}, values['action'], int(values['conversion']))
replay = sequential_replay(test, replay_policy, seed=2026)
replay_summary = {
    'thompson_replay_value_ips': replay.estimated_value,
    'lift_vs_baseline': replay.estimated_value - baseline_value,
    'matched_rows': replay.matched_rows,
    'exploration_rate': replay.exploration_rate,
    'action_counts': replay.action_counts,
}
replay_summary


In [ ]:
frozen = ThompsonSamplingPolicy(seed=2026)
for row in train.itertuples(index=False):
    values = row._asdict()
    frozen.update({column: values[column] for column in FEATURE_COLUMNS}, values['action'], int(values['conversion']))
frozen_value, matched = policy_value(test, lambda context: frozen.greedy_action(context))
comparison = pd.DataFrame([
    {'policy': 'baseline_no_email', 'value_ips': baseline_value},
    {'policy': 'best_fixed_reference', 'value_ips': best_fixed_value},
    {'policy': 'thompson_frozen', 'value_ips': frozen_value},
])
comparison['difference_vs_baseline'] = comparison['value_ips'] - baseline_value
comparison


## Interpretação e limites

O baseline principal representa uma regra fixa conservadora. A melhor ação fixa global é apresentada como referência adicional para não esconder a comparação mais difícil. IPS usa a propensão de um terço; linhas casadas e incerteza precisam acompanhar qualquer conclusão.


In [ ]:
golden_set = pd.DataFrame([
    {'recency': 2, 'history': 90, 'mens': 0, 'womens': 0, 'newbie': 1, 'zip_code': 'Urban', 'channel': 'Web'},
    {'recency': 3, 'history': 900, 'mens': 1, 'womens': 1, 'newbie': 0, 'zip_code': 'Rural', 'channel': 'Multichannel'},
    {'recency': 8, 'history': 250, 'mens': 1, 'womens': 0, 'newbie': 0, 'zip_code': 'Surburban', 'channel': 'Phone'},
    {'recency': 5, 'history': 180, 'mens': 0, 'womens': 1, 'newbie': 1, 'zip_code': 'Urban', 'channel': 'Web'},
    {'recency': 11, 'history': 1200, 'mens': 1, 'womens': 0, 'newbie': 0, 'zip_code': 'Rural', 'channel': 'Phone'},
])
golden_set['recommended_action'] = [frozen.greedy_action(row.to_dict()) for _, row in golden_set.iterrows()]
golden_set
